In [2]:
from math import exp
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

load_dotenv()

True

In [6]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0, # ตั้งค่าเป็น 0 เพื่อความแม่นยำสูงสุด
    model_kwargs={
        "logprobs": True # เปิดใช้งาน logprobs
    }
)

In [4]:
import json

def check_allergy_system(menu_data, allergy_details):
    menu_name = menu_data.get("menu_name", "")
    menu_description = menu_data.get("menu_description", "")

    # ตัดข้อมูลที่ไม่จำเป็นออก เหลือเฉพาะชื่อ allergen ให้ LLM วิเคราะห์
    allergy_names = [
        item.get("allergy_name", "").strip()
        for item in allergy_details
        if item.get("allergy_name")
    ]

    prompt = f"""ตรวจสอบเมนูอาหารว่ามีความเสี่ยงต่อสารก่อภูมิแพ้หรือไม่

menu_name: {menu_name}
menu_description: {menu_description}
allergy_names: {json.dumps(allergy_names, ensure_ascii=False)}

ให้ตอบเป็น JSON เท่านั้นในรูปแบบนี้:
{{
  "is_risky": true หรือ false,
  "matched_allergy_names": ["ชื่อสารก่อภูมิแพ้ที่พบ"],
  "self_check": {{
    "is_consistent": true หรือ false,
    "reason": "เหตุผลสั้นๆ กระชับ"
  }}
}}"""

    response = llm.invoke([HumanMessage(content=prompt)])

    try:
        parsed = json.loads(response.content)
    except json.JSONDecodeError:
        parsed = {
            "is_risky": False,
            "matched_allergy_names": [],
            "self_check": {"is_consistent": False, "reason": "parse JSON ไม่สำเร็จ"},
        }

    try:
        logprob_data = response.response_metadata.get("logprobs", {}).get("content", [])
        if logprob_data:
            chosen_logprob = logprob_data[0]["logprob"]
            confidence = exp(chosen_logprob)
        else:
            confidence = -1.0
    except (KeyError, IndexError, TypeError):
        confidence = -1.0

    matched_names = [name.strip() for name in parsed.get("matched_allergy_names", []) if isinstance(name, str)]

    # map กลับไปเป็น allergy_id เฉพาะตัวที่ LLM ระบุ
    matched_allergies = [
        {
            "allergy_id": item.get("allergy_id"),
            "allergy_name": item.get("allergy_name"),
            "count": item.get("count", 1),
        }
        for item in allergy_details
        if item.get("allergy_name") in matched_names
    ]

    if 0.1 <= confidence <= 0.9:
        status = "MANUAL_REVIEW"
    elif bool(parsed.get("is_risky")):
        status = "ALLERGY_WARN"
    else:
        status = "SAFE"

    self_check = parsed.get("self_check", {}) if isinstance(parsed, dict) else {}
    reason = ""
    if isinstance(self_check, dict):
        reason = str(self_check.get("reason", "")).strip()

    return {
        "menu_name": menu_name,
        "menu_description": menu_description,
        "matched_allergies": matched_allergies,
        "confidence": f"{confidence:.2%}" if confidence >= 0 else "N/A",
        "status": status,
        "raw_llm": response.content,
        "reason": reason,
    }

In [7]:
allergy_details = [
    {
        "allergy_id": "cc0a08b9-6879-4b1e-adcd-b1cf2817fa47",
        "allergy_name": "กลูเตน",
        "count": 1,
    },
    {
        "allergy_id": "85549d41-e901-4eed-a880-34a6681b2078",
        "allergy_name": "กุ้ง",
        "count": 1,
    },
    {
        "allergy_id": "bb48b1c3-c79a-486d-974a-d244df7577c8",
        "allergy_name": "นม",
        "count": 1,
    },
]

menu_data = {
    "menu_name": "คาเปรเซ่สลัด",
    "menu_description": "มะเขือเทศสด, ใบโหระพา, มอสซาเรลล่ายี่ห้อ A, น้ำมันมะกอก, บัลซามิก",
}

result = check_allergy_system(menu_data, allergy_details)
print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "menu_name": "คาเปรเซ่สลัด",
  "menu_description": "มะเขือเทศสด, ใบโหระพา, มอสซาเรลล่ายี่ห้อ A, น้ำมันมะกอก, บัลซามิก",
  "matched_allergies": [
    {
      "allergy_id": "bb48b1c3-c79a-486d-974a-d244df7577c8",
      "allergy_name": "นม",
      "count": 1
    }
  ],
  "confidence": "88.08%",
  "status": "MANUAL_REVIEW",
  "raw_llm": "{\n  \"is_risky\": true,\n  \"matched_allergy_names\": [\"นม\"],\n  \"self_check\": {\n    \"is_consistent\": true,\n    \"reason\": \"มีมอสซาเรลล่าซึ่งทำจากนม\"\n  }\n}",
  "reason": "มีมอสซาเรลล่าซึ่งทำจากนม"
}
